In [8]:
import pandas as pd
import requests
import selenium
from bs4 import BeautifulSoup
import time

In [9]:
#test code
BASE_URL = "https://web-scraping.dev/testimonials"

def get_Element(by_tag, soup):
    return soup.select_one(by_tag) if soup else None

def parseRatings(rating):
    return len(rating.select("svg")) if rating else None

In [10]:
#test code
with requests.Session() as session:
    try:
        response = session.get(BASE_URL)
    except requests.RequestException as e:
        print(f"[Warning] Request failed for {BASE_URL}: {e}")

    soup = BeautifulSoup(response.text, "html.parser")
    testimonials = soup.select(".testimonial")

    for testimonial in testimonials:
        rating = get_Element(".rating", testimonial)
        numerical_rating = parseRatings(rating) #this is the number of stars

        text = get_Element(".text", testimonial).get_text(strip=True)

        print(text)

We've been using this utility for years - awesome service!
This Python app simplified my workflow significantly. Highly recommended.
Had a few issues at first, but their support team is top-notch!
A fantastic tool - it has everything you need and more.
The interface could be a little more user-friendly.
Been a fan of this app since day one. It just keeps getting better!
The recent updates really improved the overall experience.
A decent web app. There's room for improvement though.
The app is reliable and efficient. I can't imagine my day without it now.
Encountered some bugs. Hope they fix it soon.


In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [18]:
def count_increased(previous_count):
    def condition(driver):
        current = len(driver.find_elements(By.CSS_SELECTOR, ".testimonial"))
        return current if current > previous_count else False
    return condition

def get_Driver_Elements(by_tag, driver):
    return driver.find_elements(By.CSS_SELECTOR, by_tag) if driver else None
    
def get_Driver_Element(by_tag, driver):
    return driver.find_element(By.CSS_SELECTOR, by_tag) if driver else None

In [19]:
driver = webdriver.Chrome()
driver.get(BASE_URL)

all_testimonials = {}
batch = 0

cards = get_Driver_Elements(".testimonial", driver)
for card in cards:
    text = get_Driver_Element(".text", card).text.strip()
    if text not in all_testimonials:
        rating = len(get_Driver_Elements(".rating svg", card))
        all_testimonials[text] = {"text": text, "rating": rating, "batch": batch}

while True:
    print(f"Querying Batch {batch}")
    previous_count = len(driver.find_elements(By.CSS_SELECTOR, ".testimonial"))
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    try:
        WebDriverWait(driver, 5).until(count_increased(previous_count))
    except TimeoutException:
        break

    batch += 1
    cards = get_Driver_Elements(".testimonial", driver)
    for card in cards:
        text = get_Driver_Element(".text", card).text.strip()
        if text not in all_testimonials:
            rating = len(get_Driver_Elements(".rating svg", card))
            all_testimonials[text] = {"text": text, "rating": rating, "batch": batch}

driver.quit()

Querying Batch 0
Querying Batch 1
Querying Batch 2
Querying Batch 3
Querying Batch 4
Querying Batch 5


In [20]:
df = pd.DataFrame(list(all_testimonials.values()))
before = len(df)
df = df.drop_duplicates(subset="text", keep="first")
after = len(df)
print(f"[Dedup] {before - after} duplicate(s) removed")

[Dedup] 0 duplicate(s) removed


In [23]:
df.to_csv("23L_0662_versionC_dynamic_testimonials.csv", index=False, encoding="utf-8")
print("[Saved] 23L_0662_versionC_dynamic_testimonials.csv")

[Saved] 23L_0662_versionC_dynamic_testimonials.csv


# Scraping Methodology (Dynamic Scraping)
- I used the developer tools on mozilla firefox. There is the option of selecting an element and displaying its code
- In the code I presume. I navigated by scrolling using selenium. It opens an instance of chrome. so I did that.
- I used the standard WebDriverWait for selenium so that it allows the chrome instance to actually load the new testimonials.
- WebDriverWait waits for 5 seconds for new testimonials to appear... if they dont then it exits
- Naah this one was actually pretty easy.
- Visual Check for correctness. And cross check for Counts

# Scraping Statistics

In [26]:
print(f"Total Batches/Scrolls: {batch}")
print(f"Total Number of Records: {after}")
print(f"Total Number of Records Extracted: {len(all_testimonials)}")
print(f"Number of Duplicate Records: {before- after}")
print(f"Number of Missing Records: {after - before}")

Total Batches/Scrolls: 5
Total Number of Records: 60
Total Number of Records Extracted: 60
Number of Duplicate Records: 0
Number of Missing Records: 0
